In [ ]:
pip install opencv-python

In [ ]:
pip install inference-sdk

In [ ]:
import cv2
import os
from inference_sdk import InferenceHTTPClient
from dotenv import load_dotenv

# --- LOAD ENVIRONMENT VARIABLES ---
# To wczyta zmienne z pliku .env (jeśli go znajdzie - czyli lokalnie).
# Na GitHubie pliku .env nie będzie, więc system użyje GitHub Secrets.
load_dotenv()

# --- CONFIGURATION ---
# Use /content/ paths to work safely in Google Colab (zmień na lokalne, jeśli odpalasz u siebie)
INPUT_DIR = "/input_images"
OUTPUT_DIR = "/output_images"

# Pobieranie klucza ze zmiennych środowiskowych!
API_KEY = os.getenv("ROBOFLOW_API_KEY")

# Zabezpieczenie: jeśli klucz się nie wczyta, skrypt wyrzuci błąd i się zatrzyma
if not API_KEY:
    raise ValueError("Błąd: Nie znaleziono klucza API! Upewnij się, że masz plik .env lub ustawione GitHub Secrets.")

MODEL_ID = "roi_detection-82ro3/3"

# Make sure the output folder exists
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- 1. INITIALIZE THE ROBOFLOW CLIENT ---
CLIENT = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key=API_KEY
)

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png')

# --- 2. LOOP THROUGH ALL FILES IN THE FOLDER ---
print(f"Scanning folder: {INPUT_DIR}")

for filename in os.listdir(INPUT_DIR):
    if not filename.lower().endswith(VALID_EXTENSIONS):
        continue

    input_path = os.path.join(INPUT_DIR, filename)
    print(f"\nProcessing: {filename}...")

    # --- 3. INFERENCE ---
    try:
        api_result = CLIENT.infer(input_path, model_id=MODEL_ID)
    except Exception as e:
        print(f"API error for file {filename}: {e}")
        continue

    # --- 4. LOAD AND CROP THE IMAGE ---
    image = cv2.imread(input_path)

    if image is None:
        print(f"Error: Failed to load image from path: {input_path}")
        continue

    # Get the dimensions of the original image (height, width)
    img_height, img_width = image.shape[:2]

    # Use enumerate to get the index 'i' (0, 1, 2...) for each detected box
    for i, prediction in enumerate(api_result['predictions']):
        x_center = prediction['x']
        y_center = prediction['y']
        width = prediction['width']
        height = prediction['height']

        # Calculate corner coordinates (cast to int)
        x1 = int(x_center - (width / 2))
        y1 = int(y_center - (height / 2))
        x2 = int(x_center + (width / 2))
        y2 = int(y_center + (height / 2))

        # Safety check: the box must not go outside the image boundaries
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_width, x2)
        y2 = min(img_height, y2)

        # If for some reason the coordinates are invalid (width/height <= 0), skip
        if x2 <= x1 or y2 <= y1:
            continue

        # CROPPING: select pixels from y1 to y2 and from x1 to x2
        cropped_image = image[y1:y2, x1:x2]

        # --- 5. SAVE THE CROP ---
        # Split the filename into name and extension (e.g. 'photo' and '.jpg')
        name, ext = os.path.splitext(filename)
        # Create a new filename with the detection index, e.g. 'photo_crop_0.jpg'
        crop_filename = f"{name}_crop_{i}{ext}"
        output_path = os.path.join(OUTPUT_DIR, crop_filename)

        cv2.imwrite(output_path, cropped_image)
        print(f" Saved crop: {output_path}")

print("\nFinished processing and cropping all images!")